Imports

In [10]:
from datasets import load_dataset
from collections import Counter
import gc
import ast

Get the dataset

In [2]:
danbooru_metadata = load_dataset("trojblue/danbooru2025-metadata", split="train")
# it will occupy 20GB, please be careful
df = danbooru_metadata.to_pandas()

Resolving data files:   0%|          | 0/43 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/43 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/43 [00:00<?, ?it/s]

In [4]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 9113285 entries, 0 to 9113284
Data columns (total 59 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   file_url                  str    
 1   approver_id               float64
 2   bit_flags                 int64  
 3   created_at                str    
 4   down_score                int64  
 5   fav_count                 int64  
 6   file_ext                  str    
 7   file_size                 int64  
 8   has_active_children       bool   
 9   has_children              bool   
 10  has_large                 bool   
 11  has_visible_children      bool   
 12  image_height              int64  
 13  image_width               int64  
 14  is_banned                 bool   
 15  is_deleted                bool   
 16  is_flagged                bool   
 17  is_pending                bool   
 18  large_file_url            str    
 19  last_comment_bumped_at    str    
 20  last_commented_at         str    
 

In [5]:
# =========================
# Filtering parameters
# =========================

MIN_SCORE = 10

MIN_FILE_SIZE = 50 * 1024          # 50 KB
MAX_FILE_SIZE = 20 * 1024 * 1024   # 20 MB

MIN_WIDTH = 224
MIN_HEIGHT = 224

# =========================
# Filter
# =========================

cleaned_df = df[
    (~df["is_deleted"]) &
    (~df["is_banned"]) &
    (~df["is_flagged"]) &
    (~df["is_pending"]) &
    (df["score"] >= MIN_SCORE) &
    (df["media_asset_file_size"] >= MIN_FILE_SIZE) &
    (df["media_asset_file_size"] <= MAX_FILE_SIZE) &
    (df["image_width"] >= MIN_WIDTH) &
    (df["image_height"] >= MIN_HEIGHT)
].copy()

# =========================
# Keep only useful columns
# =========================

cleaned_df = cleaned_df[
    [
        "id",
        "media_asset_created_at",
        "score",
        "rating",
        "image_width",
        "image_height",
        "media_asset_file_size",
        "file_url",
        "large_file_url",
        "media_asset_variants",
        "tag_string_general",
        "tag_string_artist",
        "tag_string_character",
        "tag_string_copyright",
    ]
].reset_index(drop=True)

In [ ]:
del df
gc.collect()

In [8]:
cleaned_df

,id,media_asset_created_at,score,rating,image_width,image_height,media_asset_file_size,file_url,large_file_url,media_asset_variants,tag_string_general,tag_string_artist,tag_string_character,tag_string_copyright
0,9158782,2025-04-14T21:27:10.205-04:00,10,s,3277,4096,793539,https://cdn.donmai.us/original/99/d3/99d3353f7...,https://cdn.donmai.us/sample/99/d3/sample-99d3...,"{'type': '180x180', 'url': 'https://cdn.donmai...",1girl alternate_costume asymmetrical_bangs bel...,truejekart,gallica_(metaphor:_refantazio),metaphor:_refantazio
1,9158775,2025-04-16T01:00:34.544-04:00,11,s,1288,2048,227474,https://cdn.donmai.us/original/e0/ab/e0abbf3a0...,https://cdn.donmai.us/sample/e0/ab/sample-e0ab...,"{'type': '180x180', 'url': 'https://cdn.donmai...",1girl ahoge black_choker black_nails bra breas...,lazik_1337,nanashi_mumei,hololive hololive_english
2,9158765,2025-04-16T00:40:12.485-04:00,28,e,2105,2834,1883256,https://cdn.donmai.us/original/be/9e/be9e0e4c8...,https://cdn.donmai.us/sample/be/9e/sample-be9e...,"{'type': '180x180', 'url': 'https://cdn.donmai...",1boy 2girls animal_ears aqua_panties banknote ...,eufoniuz,oguri_cap's_mother_(umamusume) oguri_cap_(umam...,umamusume umamusume:_cinderella_gray
3,9158764,2025-04-16T00:56:30.410-04:00,14,s,2907,5000,16431649,https://cdn.donmai.us/original/52/2b/522b0e2cf...,https://cdn.donmai.us/sample/52/2b/sample-522b...,"{'type': '180x180', 'url': 'https://cdn.donmai...",1girl black_hair black_skirt blue_sweater blus...,kiyoi_(gyhw8444),karin_(blue_archive) karin_(school_uniform)_(b...,blue_archive
4,9158763,2025-04-14T21:31:05.775-04:00,10,s,1447,2047,155672,https://cdn.donmai.us/original/41/ab/41abfac1e...,https://cdn.donmai.us/sample/41/ab/sample-41ab...,"{'type': '180x180', 'url': 'https://cdn.donmai...",1girl :o ahoge bent_over between_breasts black...,philo_324,yumia_liessfeldt,atelier_(series) atelier_yumia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4795193,5,2005-05-23T23:57:53.000-04:00,83,q,512,384,62853,https://cdn.donmai.us/original/4f/5a/4f5ac0caa...,https://cdn.donmai.us/original/4f/5a/4f5ac0caa...,"{'type': '180x180', 'url': 'https://cdn.donmai...",3girls :d ass bamboo_fence barefoot bath bathi...,sanshita,kojima_kirie koyomi_hare_nanaka miharu_sena_ka...,girls_bravo
4795194,4,2005-05-23T23:42:26.000-04:00,36,s,600,730,122693,https://cdn.donmai.us/original/55/2e/552e74406...,https://cdn.donmai.us/original/55/2e/552e74406...,"{'type': '180x180', 'url': 'https://cdn.donmai...",2girls :d ;) arm_up armpits bare_arms bare_sho...,soshina_nohito,sugiura_midori tokiha_mai,my-hime
4795195,3,2005-05-23T23:38:05.000-04:00,53,g,444,621,190709,https://cdn.donmai.us/original/fd/b4/fdb47f79f...,https://cdn.donmai.us/original/fd/b4/fdb47f79f...,"{'type': '180x180', 'url': 'https://cdn.donmai...",1girl 2005 ahoge armor armored_dress blonde_ha...,nakamura_hisashi,artoria_pendragon_(fate) saber_(fate),fate/stay_night fate_(series)
4795196,2,2005-05-23T23:37:30.000-04:00,26,s,450,450,232601,https://cdn.donmai.us/original/71/0f/710fd9cba...,https://cdn.donmai.us/original/71/0f/710fd9cba...,"{'type': '180x180', 'url': 'https://cdn.donmai...",2000s_(style) 2boys 2girls ahoge ass blue_hair...,nakamura_hisashi,card_master_peach haga_reiko kuhonbutsu_taishi...,cardcaptor_sakura comic_party leaf_(studio) to...


In [20]:
cleaned_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 4660225 entries, 0 to 4660224
Data columns (total 9 columns):
 #   Column                  Dtype
---  ------                  -----
 0   id                      int64
 1   media_asset_created_at  str  
 2   score                   int64
 3   rating                  str  
 4   media_asset_file_size   int64
 5   tag_string_general      str  
 6   tag_string_character    str  
 7   tag_string_copyright    str  
 8   image_url               str  
dtypes: int64(3), str(6)
memory usage: 2.5 GB


In [11]:
# Get all images with 360x360 variant
def get_360_url(variants):
    if not isinstance(variants, str):
        return None

    try:
        variants = ast.literal_eval(f"[{variants}]")
    except Exception:
        return None

    for variant in variants:
        if variant.get("type") == "360x360":
            return variant.get("url")

    return None


cleaned_df["image_url"] = cleaned_df["media_asset_variants"].map(get_360_url)

# Keep only images that actually have a 360x360 variant
cleaned_df = cleaned_df[cleaned_df["image_url"].notna()].copy()

# Drop the bulky column
cleaned_df = cleaned_df.drop(columns=["media_asset_variants", "file_url", "large_file_url"]).reset_index(drop=True)
cleaned_df = cleaned_df.drop(columns=["image_width", "image_height", "tag_string_artist"]).reset_index(drop=True)

In [19]:
cleaned_df

,id,media_asset_created_at,score,rating,media_asset_file_size,tag_string_general,tag_string_character,tag_string_copyright,image_url
0,9158782,2025-04-14T21:27:10.205-04:00,10,s,793539,1girl alternate_costume asymmetrical_bangs bel...,gallica_(metaphor:_refantazio),metaphor:_refantazio,https://cdn.donmai.us/360x360/99/d3/99d3353f73...
1,9158775,2025-04-16T01:00:34.544-04:00,11,s,227474,1girl ahoge black_choker black_nails bra breas...,nanashi_mumei,hololive hololive_english,https://cdn.donmai.us/360x360/e0/ab/e0abbf3a04...
2,9158765,2025-04-16T00:40:12.485-04:00,28,e,1883256,1boy 2girls animal_ears aqua_panties banknote ...,oguri_cap's_mother_(umamusume) oguri_cap_(umam...,umamusume umamusume:_cinderella_gray,https://cdn.donmai.us/360x360/be/9e/be9e0e4c87...
3,9158764,2025-04-16T00:56:30.410-04:00,14,s,16431649,1girl black_hair black_skirt blue_sweater blus...,karin_(blue_archive) karin_(school_uniform)_(b...,blue_archive,https://cdn.donmai.us/360x360/52/2b/522b0e2cf1...
4,9158763,2025-04-14T21:31:05.775-04:00,10,s,155672,1girl :o ahoge bent_over between_breasts black...,yumia_liessfeldt,atelier_(series) atelier_yumia,https://cdn.donmai.us/360x360/41/ab/41abfac1ef...
...,...,...,...,...,...,...,...,...,...
4660220,5,2005-05-23T23:57:53.000-04:00,83,q,62853,3girls :d ass bamboo_fence barefoot bath bathi...,kojima_kirie koyomi_hare_nanaka miharu_sena_ka...,girls_bravo,https://cdn.donmai.us/360x360/4f/5a/4f5ac0caa6...
4660221,4,2005-05-23T23:42:26.000-04:00,36,s,122693,2girls :d ;) arm_up armpits bare_arms bare_sho...,sugiura_midori tokiha_mai,my-hime,https://cdn.donmai.us/360x360/55/2e/552e74406a...
4660222,3,2005-05-23T23:38:05.000-04:00,53,g,190709,1girl 2005 ahoge armor armored_dress blonde_ha...,artoria_pendragon_(fate) saber_(fate),fate/stay_night fate_(series),https://cdn.donmai.us/360x360/fd/b4/fdb47f79fb...
4660223,2,2005-05-23T23:37:30.000-04:00,26,s,232601,2000s_(style) 2boys 2girls ahoge ass blue_hair...,card_master_peach haga_reiko kuhonbutsu_taishi...,cardcaptor_sakura comic_party leaf_(studio) to...,https://cdn.donmai.us/360x360/71/0f/710fd9cba4...


In [ ]:
!mkdir -p data

In [21]:
cleaned_df.to_parquet(
    "data/danbooru2025_cleaned.parquet",
    index=False,
    compression="zstd",
)